# 4장 1강: LLM API 활용 — 실습문제


## 실습 목표

- LLM API의 역할과 입력 준비·요청·처리·응답·확인 흐름을 설명할 수 있다.
- `model`, `instructions`, `input`의 역할을 구분하여 요청 정보를 구성할 수 있다.
- 응답의 `id`, `status`, `output`, `usage`를 읽을 수 있다.
- 요청 처리 성공, 결과 존재, 출력 형식, 분석 정확성을 구분하여 검증할 수 있다.
- 원문·기존 출력·새 응답을 구분하고 실패 상태와 확인된 사용량을 관리할 수 있다.
- 인증 정보, 오류 원인, 비용, 입력 데이터 활용 시 확인할 사항을 설명할 수 있다.

## 실습 환경 / 데이터

- 기본 실습: Python 3, Jupyter Notebook 또는 VS Code 노트북, pandas
- 세 CSV를 노트북과 같은 폴더에 둡니다. 다른 위치라면 `DATA_DIR`를 수정합니다.
- 이전 노트북을 실행할 필요는 없습니다. 이 파일의 준비 코드부터 순서대로 실행합니다.

| 파일 | 활용 |
|---|---|
| 엘지유플러스_분류.csv | 상담 요건 분류의 입력·지시·기존 출력을 확인하고 API 요청 구성 |
| 엘지유플러스_요약.csv | 요약 응답이 원문과 출력 조건을 충족하는지 검토 |
| 엘지유플러스_질의응답.csv | 특정 사실을 묻는 요청과 답변을 원문에 대조 |

세 파일의 공통 컬럼은 `source_id`, `consulting_category`, `task_category`, `instruction`, `input`, `output`입니다. `source_id`는 상담 원문 ID이며, 같은 상담에 여러 작업·질문이 연결되므로 파일과 세부 작업도 확인합니다. `instruction`은 기존 지시, `input`은 원문, `output`은 파일에 이미 저장된 출력입니다.

### 기본 실습과 실제 호출 구분

교안의 요청·응답 구조와 결과 확인을 먼저 익힐 수 있도록 **필수·과제는 가상 응답으로 실행**합니다. 노트북 끝에는 같은 내용을 실제 API 단일 요청에 연결하는 **선택 실행용 제공 코드**가 있습니다.

- 원문·지시·기존 출력은 실제 CSV 값입니다. 기존 출력도 원문과 비교할 참고 자료입니다.
- 기본 실습의 응답 ID·응답 텍스트·HTTP 코드·사용 토큰 수는 작성자가 정한 가상 값입니다. 실제 모델 실행·토큰 측정·결제 결과가 아닙니다.
- `model`의 `사용 가능한 모델 ID`는 구조 설명용 표시입니다. 실제 호출에서는 본인 계정에서 사용할 수 있는 모델 ID로 바꿉니다.
- 실제 호출은 기본적으로 꺼져 있습니다. 선택 실행에는 `openai` 패키지와 사용 가능한 API 키·모델이 필요하며, 상담 원문이 외부 API로 전송되고 비용이 발생할 수 있습니다.
- API 키는 코드·출력·공유 파일에 적지 않습니다. 환경변수 또는 선택 코드의 비공개 입력을 사용합니다.
- 상세 일괄 처리 파이프라인, 자동 재시도 로직, 구조화된 출력 강제, 요금표 기반 비용 계산은 요구하지 않습니다.
- `<NAME>`, `<DATE>`, `<CHARGE>` 같은 가림 표시는 그대로 유지합니다. 실습의 원문 해석을 현재 통신사 정책으로 일반화하지 않습니다.

**난이도와 구성:** 기초~중급, 필수 2문제와 같은 수준의 과제 1문제입니다. 제공 함수의 구현을 암기하기보다 요청과 응답을 읽고 판단하는 데 집중합니다.


## 시작 전에 읽는 개념 정리

### 1. 이번 강의에서 배우는 연결 관계

앞 강의에서는 원문과 지시를 어떻게 작성하고 결과를 어떻게 검토할지 살펴보았습니다. 이번에는 **파이썬 프로그램이 지시와 원문을 API에 전달하고, 돌아온 응답을 표에 연결하는 과정**을 배웁니다. API는 프로그램끼리 정해진 방식으로 요청과 응답을 주고받는 연결 방법입니다. LLM API는 그 연결을 통해 언어 모델의 기능을 프로그램에서 사용할 수 있게 합니다.

대화창에서는 사람이 내용을 입력하고 답을 읽습니다. API를 이용하면 프로그램이 데이터를 읽어 요청을 구성하고, 응답의 상태·텍스트·사용량을 받아 후속 처리에 연결할 수 있습니다. **API를 사용한다는 사실이 생성 내용의 정확성까지 보장하지는 않습니다.** 원문과 작업 조건을 바탕으로 검토하는 절차가 여전히 필요합니다.

| 흐름 | 실제 호출에서의 의미 | 이번 기본 실습에서 하는 일 |
|---|---|---|
| 입력 준비 | 원문과 작업 지시를 정리 | CSV에서 한 작업 행을 선택 |
| 요청 | 모델·지시·원문을 API에 전달 | 요청 딕셔너리를 구성하되 전송하지 않음 |
| LLM 처리 | 서비스가 요청을 처리하고 결과 생성 | 실제 모델 처리는 하지 않음 |
| 응답 | 상태·출력·사용량 등의 정보 반환 | 제공 함수로 가상 응답 구조를 만듦 |
| 결과 확인 | 처리 상태와 생성 내용을 점검 | 가상 응답을 읽고 원문·조건과 대조 |

따라서 기본 모드에서 정답 코드가 실행되었다는 말은 **데이터 조회와 가상 응답 점검 코드가 실행되었다**는 뜻입니다. 모델을 호출하여 정답을 얻었다는 뜻으로 해석하지 않습니다.

### 2. 요청과 응답에서 처음 만나는 용어

| 용어 | 쉬운 설명 | 이번 코드에서 읽는 위치 |
|---|---|---|
| 요청 | 프로그램이 서비스에 보내려는 작업 정보 | `request_body`, `qa_request`, `summary_request` |
| 모델 ID | 어떤 모델로 처리할지 지정하는 이름 | 요청의 `model` |
| 지시 | 해야 할 작업과 지켜야 할 조건 | 요청의 `instructions` |
| 입력 본문 | 분석 대상이 되는 상담 원문 | 요청의 `input` |
| API Key | 서비스 이용을 인증하는 비밀 정보 | 실제 호출용 별도 설정 |
| 응답 | 요청 처리 후 돌아오는 정보 묶음 | 기본 모드의 `demo_response` |
| 응답 ID | 특정 응답을 구별하는 식별 정보 | 응답의 `id` |
| 처리 상태 | 응답 처리의 진행·완료 상태를 나타내는 정보 | 응답의 `status` |
| 생성 출력 | 모델이 생성한 결과에 해당하는 부분 | 응답의 `output` 안의 텍스트 |
| 사용량 | 처리한 입력·출력 등의 토큰 수 | 응답의 `usage` |
| HTTP 상태 코드 | 요청 처리 결과를 전달하는 통신상의 코드 | 과제 표의 `HTTP코드` |
| SDK | API를 프로그램에서 쓰도록 돕는 도구 묶음 | 선택 실행의 `OpenAI` |

`HTTP코드`와 `status`는 서로 다른 정보입니다. 과제에서는 HTTP 200과 응답 상태 completed를 함께 제시하지만, 같은 필드를 두 번 적은 것이 아닙니다. 기본 실습의 가상 응답 함수에는 HTTP 코드가 들어 있지 않습니다. 필수 2의 HTTP 200은 문제에서 정한 상황이고, 과제의 HTTP 코드는 따로 만든 처리 기록입니다.

### 3. 파이썬에서 요청·응답을 담는 방법

딕셔너리는 **이름과 값의 짝**을 저장합니다. `request_body["model"]`은 딕셔너리에서 `model`이라는 이름에 연결된 값을 꺼냅니다. 목록은 여러 값을 순서대로 담습니다. 응답의 `output`은 목록이고, 그 안의 각 항목과 `content` 안의 내용은 다시 딕셔너리로 표현됩니다. 하나의 응답이 여러 층으로 구성될 수 있다는 점을 먼저 이해하면 제공 함수를 읽기 쉽습니다.

JSON은 이런 이름·값·목록 구조를 표현하는 데이터 형식입니다. 파이썬 딕셔너리와 비슷하게 보이지만 같은 객체는 아닙니다. `json.dumps()`는 딕셔너리를 JSON 모양의 문자열로 바꾸어 보여 주는 데 사용합니다. 이 함수만 실행한다고 API로 요청이 전송되지는 않습니다.

### 4. 네 가지 확인 질문을 구분하기

| 확인 단계 | 스스로 물어볼 질문 | 이 단계만으로는 알 수 없는 것 |
|---|---|---|
| 요청·처리 상태 | 요청이 처리되었고 필요한 응답 상태인가? | 텍스트가 실제로 있는지, 내용이 맞는지 |
| 결과 존재 | 비어 있지 않은 결과가 있는가? | 요청한 형식인지, 원문과 일치하는지 |
| 출력 형식 | 범주명·문장 수·길이 등 조건을 지켰는가? | 선택한 범주나 사실이 정확한지 |
| 분석 정확성 | 원문과 분류 기준에 맞는 내용인가? | 다른 요청도 모두 정확하다고 일반화할 수 있는지 |

`True`라는 출력도 **어떤 조건을 검사한 결과인지** 함께 읽습니다. 빈 문자열인지 검사한 `True`는 ‘내용이 맞다’는 판정이 아닙니다. 형식 검사를 통과해도 내용 검토에서 수정이 필요할 수 있습니다.

### 5. 이름이 비슷해도 서로 다른 값

| 구분 | 의미 |
|---|---|
| CSV의 `output` | 파일에 이미 저장되어 있는 분류·요약·답변 |
| 가상 응답의 `output` | 메시지와 내용 항목을 담은 목록 |
| 내용 항목의 `type: output_text` | 해당 항목이 출력 텍스트임을 나타내는 표시 |
| SDK의 `response.output_text` | 생성 텍스트를 읽기 위한 편의 속성 |
| 결과 표의 `새_가상출력` / `새_API출력` | 새로 받은 것으로 다루는 응답을 기존 출력과 구분해 기록한 열 |

기존 출력이 새 응답과 같더라도 두 값의 출처는 다릅니다. 기존 답을 읽거나 복사한 일을 모델이 답한 것으로 기록하지 않습니다. 또한 본문 글자 수와 토큰 수는 다른 값이고, 토큰 수와 원화 비용도 같은 값이 아닙니다.


## 실습 준비

아래 **제공 코드**를 실행하세요.

1. 세 파일의 크기, 컬럼·자료형, 결측치·빈 문자열을 확인합니다.
2. 고유 상담 수, 작업 유형 분포와 본문 길이를 확인합니다.
3. 상담 40003·40005의 원문이 세 파일에서 동일한지 확인합니다.
4. `get_record()`로 정확한 작업 행을 선택하고 `show_record()`로 내용을 확인합니다.

`record_id`는 파일 이름과 0부터 시작하는 행 위치를 결합한 **실습용 행 식별자**입니다. 원본의 영구 ID나 API 응답 ID가 아닙니다. 본문 글자 수도 API 토큰 사용량이 아닙니다.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")
classification = pd.read_csv(DATA_DIR / "엘지유플러스_분류.csv")
summarization = pd.read_csv(DATA_DIR / "엘지유플러스_요약.csv")
qa = pd.read_csv(DATA_DIR / "엘지유플러스_질의응답.csv")
tables = {"분류": classification, "요약": summarization, "질의응답": qa}

for name, table in tables.items():
    print(f"\n[{name}] 원본 {table.shape[0]:,}행 × {table.shape[1]}열")
    print(pd.DataFrame({"자료형": table.dtypes.astype(str),
                        "결측치": table.isna().sum()}).to_string())
    print("고유 상담 ID 수:", table["source_id"].nunique())
    print("빈 본문 수:", table["input"].str.strip().eq("").sum())
    print("빈 지시 수:", table["instruction"].str.strip().eq("").sum())
    print("빈 출력 수:", table["output"].str.strip().eq("").sum())
    print("작업 유형별 행 수:")
    print(table["task_category"].value_counts().to_string())
    print("행별 본문 글자 수(토큰 수가 아님):")
    print(table["input"].str.len().agg(["min", "mean", "max"]).round(2).to_string())
    table["record_id"] = [f"{name}-{i}" for i in range(len(table))]

practice_ids = [40003, 40005]
for source_id in practice_ids:
    texts = []
    print(f"\n[연습 상담 {source_id}]")
    for name, table in tables.items():
        part = table[table["source_id"] == source_id]
        print(name, "행 수:", len(part))
        texts.extend(part["input"].tolist())
    print("세 파일에서 이 상담의 원문이 모두 동일:", pd.Series(texts).nunique() == 1)
    print("이 상담 원문의 글자 수:", len(texts[0]))

def get_record(table, source_id, task_category, instruction=None):
    selected = table[(table["source_id"] == source_id)
                     & (table["task_category"] == task_category)]
    if instruction is not None:
        selected = selected[selected["instruction"] == instruction]
    if len(selected) != 1:
        raise ValueError("한 행을 선택해야 합니다. ID·작업 유형·질문을 다시 확인하세요.")
    return selected.iloc[0]

def show_record(row, include_input=False):
    print("\nrecord_id:", row["record_id"], " / source_id:", row["source_id"])
    print("작업 유형:", row["task_category"])
    print("지시 또는 질문:", row["instruction"])
    if include_input:
        print("상담 원문:\n" + row["input"])
    print("CSV에 저장된 출력:", row["output"])


### 준비 코드 읽기 — 파일과 표의 구조부터 확인하기

`pandas`의 `DataFrame`은 행과 열로 이루어진 표입니다. 이번 데이터의 한 행은 ‘상담 한 개’ 자체보다 **특정 상담에 대한 작업 한 개**로 이해하면 좋습니다. 같은 상담에 분류·요약·질문을 여러 개 만들 수 있기 때문입니다.

| 코드·표현 | 하는 일 | 결과를 읽는 방법 |
|---|---|---|
| `Path(".")` | 현재 작업 폴더를 경로로 지정 | 데이터가 다른 폴더라면 해당 폴더로 수정 |
| `DATA_DIR / "엘지유플러스_분류.csv"` | 폴더와 파일 이름을 연결 | `/`는 여기서 경로 연결 역할 |
| `pd.read_csv(...)` | CSV를 표로 읽음 | 파일 조회이며 LLM 호출이 아님 |
| `tables = {"분류": classification, ...}` | 이름과 표를 짝지어 보관 | 세 파일에 같은 점검 절차를 적용하기 위함 |
| `tables.items()` | 이름과 표를 한 쌍씩 꺼냄 | 첫 반복에서 분류 표, 다음 반복에서 다른 표 확인 |
| `table.shape` | 표의 행 수·열 수 확인 | 행 수가 곧 고유 상담 수인 것은 아님 |
| `table.dtypes.astype(str)` | 열별 자료형 이름을 문자열로 표시 | 원본 상담 본문을 변환하는 작업이 아님 |
| `table.isna().sum()` | 열별 결측치 수 계산 | 비어 있는 값 확인 |
| `table["source_id"].nunique()` | 중복을 제외한 상담 ID 개수 계산 | 동일 ID의 작업 여러 행을 한 상담으로 셈 |
| `.str.strip().eq("").sum()` | 앞뒤 공백을 제외하면 빈 문자열인 값의 수 계산 | 결측치 검사와 별도로 내용 없는 문자열 확인 |
| `.value_counts()` | 작업 유형마다 몇 행인지 계산 | 상담 수보다 작업의 구성 확인에 초점 |
| `.str.len()` | 각 행의 본문 글자 수 계산 | 토큰 수를 계산하는 기능은 아님 |
| `.agg(["min", "mean", "max"])` | 길이의 최솟값·평균·최댓값 계산 | 파일에 있는 각 행을 기준으로 집계 |
| `.round(2)` | 계산값을 소수점 둘째 자리까지 반올림 | 본문을 자르거나 요약하는 기능이 아님 |

`print()`는 값을 화면에 보여 줍니다. `f"...{값}..."`는 문자열 중간에 값을 넣는 표기입니다. 예를 들어 `{table.shape[0]:,}`의 쉼표는 큰 행 수를 천 단위로 나누어 보여 줍니다. `.to_string(index=False)`는 표를 문자열로 출력하면서 왼쪽 인덱스를 숨기는 옵션이며, ID 열이나 실제 행을 삭제하지 않습니다.

### 같은 상담에 여러 작업이 있을 때의 ID

- `source_id`: 어느 상담에서 나온 자료인지 구분합니다.
- `task_category`: 그 상담에 대해 어떤 종류의 작업을 수행하는지 나타냅니다.
- `instruction`: 실제로 요구한 작업이나 질문 문장입니다.
- `record_id`: 현재 파일의 작업 행을 구별하기 위해 실습 코드가 추가한 이름입니다.

`[f"{name}-{i}" for i in range(len(table))]`는 현재 표의 각 행에 `분류-0`, `분류-1`처럼 이름을 붙입니다. `range(len(table))`는 0부터 행 수 직전까지의 번호를 만듭니다. 원본 표의 정렬 순서가 바뀌면 번호도 달라질 수 있으므로 원본 상담 ID와 같은 의미로 쓰지 않습니다.

준비 코드가 `record_id`를 추가한 뒤 표는 7열이 됩니다. 출력되는 ‘원본 6열’은 그 열을 추가하기 전에 확인한 값입니다. CSV 파일에 새 열을 저장하는 `to_csv()`는 실행하지 않습니다.


### 응답 구조 확인용 제공 함수

아래 함수는 **기본 실습용 보조 도구**입니다. 외부 API를 호출하지 않습니다.

- `make_demo_response()`는 교안의 축약 구조를 본뜬 가상 응답을 만듭니다. 실제 응답의 모든 필드를 재현하지 않습니다.
- `read_demo_text()`는 `output`의 각 항목에서 메시지와 `output_text` 내용을 확인해 텍스트를 읽습니다. 첫 번째 항목·첫 번째 내용에 항상 정답이 있다고 가정하지 않습니다.
- 실제 Python SDK 객체에서는 교안의 `response.output_text`로 생성 텍스트를 확인할 수 있습니다. 기본 실습의 딕셔너리 자체에는 그 편의 속성이 없습니다.


In [ ]:
def make_demo_response(response_id, text, input_tokens, output_tokens):
    # 학습용 축약 응답입니다. text=None은 출력이 없는 가정을 뜻합니다.
    items = []
    if text is not None:
        items = [{"type": "message", "role": "assistant",
                  "content": [{"type": "output_text", "text": text}]}]
    return {"id": response_id, "status": "completed", "output": items,
            "usage": {"input_tokens": input_tokens, "output_tokens": output_tokens,
                      "total_tokens": input_tokens + output_tokens}}

def read_demo_text(response_data):
    texts = []
    for item in response_data.get("output", []):
        if item.get("type") == "message":
            for content in item.get("content", []):
                if content.get("type") == "output_text":
                    texts.append(content.get("text", ""))
    return "".join(texts)


### 제공 함수 상세 설명 — 응답을 만들고 텍스트를 읽는 순서

이 함수들은 작성 연습의 평가 대상이 아니라 **응답 구조를 살펴보기 위한 제공 도구**입니다. 함수 이름, 전달하는 값, 반환값부터 연결해서 읽으면 됩니다. `def`는 함수를 정의하고, `return`은 처리 결과를 호출한 곳에 돌려줍니다. 함수 정의 셀을 실행하는 것만으로 실제 응답이 만들어지지는 않습니다. 아래 문제에서 함수를 호출할 때 값이 만들어집니다.

#### `make_demo_response()`는 무엇을 만드는가?

| 코드 | 역할 |
|---|---|
| `response_id, text, input_tokens, output_tokens` | 호출할 때 전달하는 응답 이름·텍스트·가상 입력량·가상 출력량 |
| `items = []` | 출력 항목을 담을 빈 목록 생성 |
| `if text is not None:` | 텍스트 값이 제공되었는지 확인 |
| `type: message`, `role: assistant` | 출력 항목이 assistant 메시지라는 구조를 표시 |
| `content` 안의 `type: output_text`, `text` | 텍스트 내용의 종류와 실제 문자열을 넣음 |
| `status: completed` | 연습용 완료 상태를 고정해 넣음 |
| `input_tokens + output_tokens` | 전달받은 두 숫자를 더해 가상 전체 사용량을 만듦 |

이 함수에는 입력 문장을 모델에 보내거나 토큰으로 세는 과정이 없습니다. `input_tokens`에 숫자를 전달하면 그 숫자를 그대로 저장합니다. `status`도 외부 서버에 물어본 값이 아니라 함수에 적힌 값입니다. 교안의 핵심 구조만 보여 주는 축약 예시이며 실제 API 응답의 모든 경우를 재현하지 않습니다.

#### `read_demo_text()`는 어느 부분을 읽는가?

| 순서 | 코드·조건 | 동작 |
|---:|---|---|
| 1 | `texts = []` | 찾은 텍스트들을 모을 목록 준비 |
| 2 | `response_data.get("output", [])` | output 목록을 읽고, 이름이 없으면 빈 목록 사용 |
| 3 | `item.get("type") == "message"` | 메시지에 해당하는 항목만 확인 |
| 4 | `item.get("content", [])` | 그 메시지의 내용 목록을 하나씩 읽음 |
| 5 | `content.get("type") == "output_text"` | 텍스트 출력에 해당하는 내용만 선택 |
| 6 | `texts.append(content.get("text", ""))` | 찾은 문자열을 목록 뒤에 추가 |
| 7 | `"".join(texts)` | 모은 문자열을 사이 구분자 없이 이어 반환 |

`.get(이름, 기본값)`은 딕셔너리에 그 이름이 없을 때 기본값을 쓰는 방식입니다. 이름은 있는데 값이 `None`인 경우까지 자동으로 기본값으로 바꾸는 기능은 아닙니다. 여기서는 위 함수가 만든 정해진 구조를 읽는 데 사용합니다. 실제 모든 응답 오류를 처리하는 범용 검사 함수로 해석하지 않습니다.

바깥 `for`는 출력 항목을, 안쪽 `for`는 해당 메시지의 내용을 순서대로 봅니다. 첫 번째 위치가 늘 텍스트라는 가정을 피하기 위한 구조입니다. 다만 텍스트를 잘 찾아냈다는 사실과 그 텍스트가 정답이라는 사실은 별개입니다.

#### `None`과 빈 문자열은 어떻게 다른가?

| 전달한 `text` | 가상 응답의 구조 | 텍스트를 읽은 결과 |
|---|---|---|
| 일반 문자열 | 메시지와 텍스트 내용 항목을 만듦 | 해당 문자열 |
| `None` | `output`을 빈 목록으로 둠 | 빈 문자열 `""` |
| 빈 문자열 `""` | 메시지 구조는 있지만 텍스트가 비어 있음 | 빈 문자열 `""` |

`None`은 값이 없음을 나타내는 파이썬 값이고 `""`는 길이가 0인 문자열입니다. 서로 다른 입력이지만 이 텍스트 읽기 함수에서는 둘 다 비어 있는 결과가 될 수 있습니다. 그래서 출력 존재 검사만으로 원래 어떤 구조였는지까지 알 수는 없습니다. 필요할 때는 응답 구조와 상태도 함께 확인합니다.


---
## 필수 1. API 요청 구성과 응답 항목 읽기

### 문제 1-1. 상담 분류 요청과 가상 응답 연결하기

#### 문제 설명

상담 ID **40005**의 `상담 요건`을 `단일 요건 민원`, `다수 요건 민원` 중 하나로 분류합니다. 원문에는 통화 중 대기 서비스 문제와 추가 결합 할인 상담이 함께 있습니다.

요청에는 다음 항목을 사용합니다.

| 요청 항목 | 작성할 내용 |
|---|---|
| model | 구조 확인용 문자열 `사용 가능한 모델 ID` |
| instructions | 분류 작업·두 범주 정의·하나만 선택·범주명만 출력 조건 |
| input | 선택한 상담의 원문 전체 |

이번 기준에서 단일은 중심 요청·주제가 하나인 경우, 다수는 서로 구분되는 요청·주제가 둘 이상인 경우입니다. 한 요청의 절차를 여러 번 질문한 것만으로 다수라고 보지는 않습니다.

다음 **가상 응답 값**을 사용하세요.

| 항목 | 값 |
|---|---|
| id | resp_class_demo |
| status | completed |
| 생성 텍스트 | 다수 요건 민원 |
| input_tokens | 1200 |
| output_tokens | 8 |
| total_tokens | 1208 |

#### 요구사항

1. `class_row = get_record(classification, 40005, "상담 요건")`으로 작업 행을 가져오고 원문을 확인하세요.
2. `model`, `instructions`, `input`의 세 항목을 가진 딕셔너리 `request_body`를 작성하세요. 인증 키와 CSV의 기존 정답 `output`을 본문에 넣지 않습니다.
3. `make_demo_response("resp_class_demo", "다수 요건 민원", 1200, 8)`로 `demo_response`를 만드세요. 응답 전체와 `id`, `status`, 생성 텍스트, `usage`를 각각 확인합니다.
4. 처리 완료 여부, 비어 있지 않은 텍스트 여부, 허용 범주 여부를 확인하세요. 실제 원문에 맞는지는 별도로 읽고 판단합니다.
5. `source_id`, `record_id`, `원문`, `기존_output`, `response_id`, `status`, `새_가상출력`, `input_tokens`, `output_tokens`, `total_tokens`의 한 행을 가진 `result_table`을 만드세요. 원문과 기존 출력은 덮어쓰지 않습니다.

#### 해석 질문

**Q1.** LLM API의 역할과 입력 준비부터 결과 확인까지의 흐름을 설명하세요. 직접 대화하는 방식과 무엇이 다른가요?

**Q2.** `model`, `instructions`, `input`, API Key의 역할을 구분하세요. 같은 기준으로 다른 상담을 분류하려면 무엇을 바꾸나요?

**Q3.** 응답 ID, 처리 상태, 생성 출력, 토큰 사용량은 각각 무엇을 뜻하나요? `source_id`와 응답 `id`, 원본 JSON의 `output`과 SDK의 `output_text`는 어떻게 다른가요?

**Q4.** 응답이 completed이고 허용 범주라면 정확성도 보장되나요? 이번 출력의 실제 근거와 토큰 수 1208을 해석할 때의 주의점을 설명하세요.

#### 제출 결과

- 요청·응답 확인 코드
- 원문과 기존 출력이 보존된 `result_table`
- Q1~Q4 답변


### 풀이 안내 — 요청 구성과 응답 읽기를 나누어 생각하기

먼저 준비 코드와 제공 함수 셀을 실행합니다. 그다음 상담 행을 선택하고, 선택한 행의 `input`을 요청에 연결합니다. `instruction`은 CSV 열 이름이고 `instructions`는 이번 API 요청 항목 이름이므로 철자를 구분하세요.

| 작업 | 확인할 점 |
|---|---|
| 요청 작성 | 어떤 모델인지, 어떤 작업인지, 어떤 원문인지 세 항목을 나누었는가? |
| 응답 읽기 | 응답 이름·상태·텍스트·사용량을 구분해 꺼냈는가? |
| 조건 확인 | 완료 여부·텍스트 존재·범주 조건을 따로 확인했는가? |
| 결과 연결 | 기존 출력과 새 응답을 다른 열에 남겼는가? |
| 해석 작성 | 원문의 어떤 대목이 분류 판단을 뒷받침하는가? |

`bool(문자열.strip())`는 앞뒤 공백을 제외하고 내용이 있는지 살펴보는 구문입니다. `값 in 목록`은 값이 목록의 항목 중 하나와 일치하는지 확인합니다. 두 구문 모두 원문의 의미를 이해하는 기능은 아닙니다. 해석 답변에는 **확인한 결과 → 해당 결과가 뜻하는 범위 → 원문 근거**를 연결해 적으세요.


In [ ]:
# 필수 1 코드를 작성하세요.


### 필수 1 답변 작성란

**Q1.**

- 답변:

**Q2.**

- 답변:

**Q3.**

- 답변:

**Q4.**

- 답변:


---
## 필수 2. 처리 성공과 분석 정확성 구분

### 문제 2-1. 분류·질의응답·요약의 가상 응답 검증

#### 문제 설명

다음은 모두 **HTTP 200으로 응답을 받았고 status가 completed인 가상 사례**입니다. 실제 API 처리 기록이 아닙니다.

| 사례 | 상담 ID / 작업 | 가상 응답 텍스트 |
|---|---|---|
| V1 | 40005 / 상담 요건 분류 | 단일 요건 민원 |
| V2 | 40005 / 상담 요건 분류 | 이 상담은 다수 요건 민원입니다. |
| V3 | 40003 / 결합 할인 지정 대상 질의응답 | 고객 본인 |
| V4 | 40003 / 요약 | 고객은 5G 슬림+로 변경했고 기존 결합 할인은 해지되었다. VIP 등급은 그대로 유지되었다. |
| V5 | 40003 / 결합 할인 지정 대상 질의응답 | 출력 없음 |

V1·V2의 범주와 기준은 필수 1과 같습니다. V3·V5는 `고객의 결합 할인은 누구 앞으로 되어 있나요?`에 **대상만 짧게 답하도록** 요청한 상황입니다. V4는 **2문장·300자 이내**로 요금제 변경·기존 결합 할인·멤버십 등급을 요약하도록 요청한 상황입니다.

#### 요구사항

1. 질의응답 파일에서 40003의 위 질문 행을 `qa_row`, 요약 파일에서 40003의 `길이 제한 요약` 행을 `summary_row`로 조회하세요. 원문과 저장된 출력도 확인합니다.
2. 각 작업의 지시와 원문을 구분한 `qa_request`, `summary_request`를 작성하세요. 모델 값은 필수 1과 같은 설명용 표시를 사용합니다.
3. 제공된 가상 응답을 `make_demo_response()`로 만들고 생성 텍스트를 읽으세요. V5에는 `text=None`을 사용합니다. 사용량은 검토의 핵심이 아니므로 입력은 모두 1000, 출력은 차례로 8·16·4·60·0의 **가상 값**을 사용합니다.
4. `사례`, `source_id`, `record_id`, `response_id`, `status`, `가상출력`, `결과 존재`, `형식 판단`, `내용 판단`, `활용 판단` 컬럼의 `validation_table`을 작성하세요. 존재 여부는 코드로 확인하고, 형식·내용은 기준과 원문을 읽고 작성합니다.
5. 잘못된 값을 원문과 비교해 어떻게 고칠지 설명하세요. 검토용 표에는 원래 가상 출력을 유지하고, 틀린 값을 정답으로 몰래 바꿔 기록하지 않습니다.

#### 해석 질문

**Q1.** 다섯 사례 모두 completed인데 왜 그대로 활용할 수 없나요? 요청 처리·결과 존재·출력 형식·분석 정확성을 구분하여 설명하세요.

**Q2.** V1·V2는 각각 어느 부분이 문제인가요? 허용 범주 확인과 원문 검증의 차이를 설명하세요.

**Q3.** V3·V4의 원문에 맞는 답변은 무엇인가요? V5처럼 결과가 없으면 CSV의 기존 출력을 새 모델 응답인 것처럼 채워도 되나요?

**Q4.** 원문, 기존 출력, 새 응답, 응답 ID와 작업 식별자를 따로 유지하면 어떤 검증에 도움이 되나요? 검토 표와 검증을 통과한 최종 분석 결과는 어떻게 구분해야 하나요?

#### 제출 결과

- 두 작업의 요청 구조와 원문 조회
- 가상 응답을 읽어 만든 `validation_table`
- 각 오류의 수정 방향과 Q1~Q4 답변


### 풀이 안내 — 같은 완료 상태에서도 다른 문제가 생깁니다

질의응답은 같은 상담 안에도 질문이 여러 개 있을 수 있으므로 정확한 질문 문장까지 지정합니다. 요약은 ‘어떤 사실을 반드시 포함해야 하는지’와 ‘어떤 길이·문장 수로 써야 하는지’를 함께 확인하세요.

| 검토 항목 | 읽을 내용 |
|---|---|
| 응답 구조 | status와 생성 텍스트를 각각 확인 |
| 결과 존재 | 비어 있지 않은 문자열인지 확인 |
| 분류 형식 | 범주 이름 하나만 출력했는지 확인 |
| 질의응답 내용 | 원문에서 질문의 대상이 누구인지 확인 |
| 요약 내용 | 원문의 변경 사항과 유지 사항을 구분 |
| 활용 판단 | 그대로 쓸 수 있는지, 수정·검토 또는 보류가 필요한지 작성 |

제공된 사례를 반복문으로 읽는 것은 여러 번의 API 요청이 아닙니다. 형식·내용·활용 판단은 원문을 읽은 뒤 사람이 작성합니다. 오류가 있는 문장을 지우기만 하지 말고, **어떤 부분이 왜 문제인지와 어떻게 수정할지**를 함께 남기세요. 출력이 없는 경우에는 정답을 알아도 새 모델 응답처럼 채우지 않습니다.


In [ ]:
# 필수 2 코드를 작성하세요.


### 필수 2 답변 작성란

**Q1.**

- 답변:

**Q2.**

- 답변:

**Q3.**

- 답변:

**Q4.**

- 답변:


---
## 과제 1. 요청 실패와 토큰 사용량 확인

### 문제 3-1. 가상 처리 기록에서 실패와 확인된 사용량 구분

#### 문제 설명

다음은 요청 상태를 관리하기 위한 **프로그램 측 가상 기록**입니다. API 원본 응답 JSON 전체가 아니며, 실제로 요청하지 않았습니다. L1·L2·L4는 40003, L3·L5는 40005의 해당 작업입니다. 같은 원문을 서로 다른 요청으로 기록하므로 `실습요청ID`로 구분합니다.

| 실습요청ID | 상담 ID / 작업 | HTTP 코드 | 응답 상태 | 새 결과 | 입력 / 출력 / 전체 토큰 |
|---|---|---:|---|---|---|
| L1 | 40003 / 할인 지정 대상 질의응답 | 200 | completed | 남편 | 1100 / 2 / 1102 |
| L2 | 40003 / 요약 | 401 | 응답 없음 | 없음 | 미확인 |
| L3 | 40005 / 상담 요건 분류 | 429 | 응답 없음 | 없음 | 미확인 |
| L4 | 40003 / 요약 | 500 | 응답 없음 | 없음 | 미확인 |
| L5 | 40005 / 상담 요건 분류 | 400 | 응답 없음 | 없음 | 미확인 |

응답 없음·결과 없음·미확인은 Python의 `None`으로 기록합니다. 실패한 요청의 사용량을 임의로 0으로 채우지 않습니다. 이 가정에서 L1만 응답 ID `resp_L1`이 있습니다.

#### 요구사항

1. 위 표로 `request_log`를 만드세요. `실습요청ID`, `source_id`, `record_id`, `HTTP코드`, `response_id`, `status`, `새결과`, `input_tokens`, `output_tokens`, `total_tokens`를 저장합니다. 작업 행은 필수 문제의 `qa_row`, `summary_row`, `class_row`를 사용합니다.
2. HTTP코드가 200이 아닌 요청만 선택하여 `failed_requests`로 확인하세요. 실패한 요청의 결과를 임의의 분류나 기존 정답으로 채우지 않습니다.
3. 실패한 요청별로 `먼저 확인할 사항`, `다음 조치`를 작성하세요. 모든 요청을 즉시 반복 호출하는 방식은 사용하지 않습니다.
4. `total_tokens`가 확인된 행만 선택하여 `known_usage`를 만들고, 입력·출력·전체 토큰의 **확인된 합계**를 구하세요. 미확인 사용량을 가진 요청 수도 출력하세요.
5. L1의 답이 원문에 맞는지 확인하고, 요청 5개를 모두 분석 완료로 볼 수 있는지 설명하세요.

**보조 구문:** `표[표["HTTP코드"] != 200]`, `표[표["total_tokens"].notna()]`, `표["total_tokens"].isna().sum()`

#### 해석 질문

**Q1.** 성공한 요청과 실패한 요청의 ID는 무엇인가요? 상담 ID만으로 L2와 L4를 구분할 수 있나요? 실패 상태와 새 결과를 왜 따로 관리해야 하나요?

**Q2.** 400·401·429·500은 각각 무엇을 먼저 확인해야 하나요? 인증 오류와 일시적인 서버 오류에 같은 재시도 방식을 적용하면 왜 문제가 되나요?

**Q3.** 확인된 입력·출력·전체 토큰 합계와 사용량 미확인 요청 수는 얼마인가요? 이 값을 전체 실제 사용량이나 원화 비용으로 확정할 수 있나요? 비용에 영향을 주는 요소를 설명하세요.

**Q4.** API 키가 공개 저장소에 올라갔다면 어떻게 해야 하나요? 상담 데이터를 실제 외부 API로 보내기 전에는 무엇을 확인해야 하나요?

#### 제출 결과

- `request_log`, `failed_requests`와 요청별 조치 표
- 확인된 사용량 합계와 미확인 요청 수
- 원문 근거 및 Q1~Q4 답변


### 풀이 안내 — 요청 수·결과 수·확인된 사용량을 나누기

과제는 필수 문제와 같은 수준에서 표 선택과 결과 해석을 연습합니다. 자동 재시도 함수를 만들거나 API 요청을 추가할 필요는 없습니다. 앞에서 선택한 `qa_row`, `summary_row`, `class_row`를 이용해 기록을 연결하세요.

| 구문 | 읽는 방법 |
|---|---|
| `표["HTTP코드"] != 200` | 각 행의 코드가 200과 다른지 비교 |
| `표[조건]` | 비교 결과가 True인 행만 선택 |
| `.copy()` | 선택한 결과를 별도 표로 복사 |
| `.notna()` | 결측이 아닌 값인지 확인 |
| `.isna()` | 결측값인지 확인 |
| `.sum()` | 숫자를 더하거나, True가 몇 개인지 셈 |

미확인 사용량은 ‘측정한 결과 0’과 다릅니다. 합계를 구한 뒤에는 어떤 행만 포함했는지와 빠진 행이 몇 개인지 함께 설명합니다. 오류 조치도 ‘다시 실행’ 한 줄로 통일하지 말고, 먼저 고쳐야 할 입력·인증 문제인지 또는 제한·서비스 상태를 확인할 상황인지 구분해 적으세요.


In [ ]:
# 과제 1 코드를 작성하세요.


### 과제 1 답변 작성란

**Q1.**

- 답변:

**Q2.**

- 답변:

**Q3.**

- 답변:

**Q4.**

- 답변:


### 개념 다시 정리 — 확인한 사실만큼만 결론 내리기

| 관찰한 사실 | 말할 수 있는 결론 | 추가로 필요한 확인 |
|---|---|---|
| 요청 딕셔너리를 만들었다 | 요청할 정보를 구성했다 | 실제 전송 여부와 반환 상태 |
| status가 completed이다 | 해당 응답에서 완료 상태를 확인했다 | 필요한 출력의 존재·형식·내용 |
| 텍스트 존재 검사가 True이다 | 공백만 있는 결과는 아니다 | 작업 조건과 사실 관계 |
| 허용 범주명과 일치한다 | 분류 출력의 형식 조건에 맞는다 | 올바른 범주인지 원문 대조 |
| usage 일부가 확인된다 | 그 범위의 사용량을 기록할 수 있다 | 미확인 요청과 실제 비용 정책 |
| 오류 원인과 조치를 표에 적었다 | 다음 확인 방향을 정리했다 | 실제 원인 해결과 재요청 여부 판단 |

코드가 한 작업을 끝냈을 때 무엇이 확인되었고 무엇이 아직 남았는지 구분하면 결과 해석이 명확해집니다. 실습 답변에는 데이터 출처, 확인한 조건, 근거, 남아 있는 불확실성을 연결해 적습니다. 이번에는 새로운 분석 기법이나 추가 코딩 문제를 도입하지 않고 요청·응답 확인이라는 같은 흐름을 반복해서 익힙니다.


---
## 실습 마무리

1. LLM API로 해결하려던 분석 문제와 요청·응답 흐름은 무엇인가요?
2. 원본 데이터와 새 응답을 어떻게 구분하여 연결했나요?
3. 처리 성공과 분석 정확성을 어떤 단계로 나누어 확인했나요?
4. 실패 요청과 사용량을 해석할 때 무엇을 임의로 채우면 안 되나요?
5. 이번 기본 실습에서 실제 실행한 작업과 실제로 요청하지 않은 작업은 무엇인가요?


### 실습 마무리 답변 작성란

1.

2.

3.

4.

5.


---
## 선택 실행: 단일 상담을 실제 API에 연결하기 — 제공 코드

교안의 단일 요청·응답 흐름을 확인하는 선택 실행입니다. **추가 평가 문제는 아닙니다.** 기본 필수·과제는 키 없이 끝낼 수 있습니다.

1. 실제 호출 환경은 현재 OpenAI Python SDK가 지원하는 Python 버전을 사용합니다. 필요하면 `%pip install openai`를 별도 셀에서 실행합니다.
2. 자신의 계정에서 사용할 수 있는 텍스트 모델 ID를 `LIVE_MODEL_ID`에 적습니다. 모델 ID는 비밀 키가 아닙니다.
3. API 키는 `OPENAI_API_KEY` 환경변수에 설정합니다. 없으면 코드의 비공개 입력창에 입력할 수 있습니다. 코드 셀에 키 문자열을 직접 쓰지 않습니다.
4. 데이터 활용 권한·처리 정책·개인정보를 확인하고 실제 전송하려는 경우에만 `RUN_LIVE_API = True`로 바꿉니다. 상담 40005의 원문 전체가 전송되며 과금될 수 있습니다.
5. 반환 상태·텍스트·usage를 읽고, 필수 1과 같은 네 단계로 검토합니다. 실제 응답과 토큰 수는 가상 예시와 다를 수 있습니다.

자동 재시도는 꺼 둔 제공 코드입니다. 실패하면 상태 코드 또는 연결 문제를 확인하고 원인을 해결한 뒤 필요한 경우에만 다시 실행합니다. 실행을 반복하면 같은 상담을 다시 요청하게 됩니다.

이 코드의 기본값은 False이며, 제작 과정에서는 실제 API 호출을 하지 않았습니다. 요청 구문은 [공식 텍스트 생성 안내](https://developers.openai.com/api/docs/guides/text?lang=python)와 [Python SDK 문서](https://developers.openai.com/api/reference/python)에서 확인했습니다. 교안 밖의 도구 호출·일괄 처리·출력 스키마 강제는 추가하지 않았습니다.


In [ ]:
RUN_LIVE_API = False
LIVE_MODEL_ID = ""  # 실제로 사용할 수 있는 모델 ID만 입력하세요. API 키를 쓰는 곳이 아닙니다.

if not RUN_LIVE_API:
    print("실제 API 호출을 실행하지 않았습니다. 기본 실습은 가상 응답으로 진행합니다.")
else:
    import os
    from getpass import getpass
    from openai import OpenAI, APIStatusError, APIConnectionError

    if not LIVE_MODEL_ID.strip():
        raise ValueError("사용 가능한 모델 ID를 LIVE_MODEL_ID에 지정하세요.")
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("API 키를 비공개로 입력하세요: ").strip()
    if not os.environ.get("OPENAI_API_KEY"):
        raise ValueError("API 키가 비어 있습니다. 키는 출력하지 마세요.")

    live_row = get_record(classification, 40005, "상담 요건")
    live_result = None
    client = OpenAI(max_retries=0)
    try:
        response = client.responses.create(
            model=LIVE_MODEL_ID,
            instructions=(
                "상담의 중심 요청·주제가 하나면 단일 요건 민원, 서로 구분되는 요청·주제가 "
                "두 가지 이상이면 다수 요건 민원으로 분류하세요. 같은 요청의 세부 절차 질문만 "
                "이어지면 하나로 봅니다. 두 범주 중 하나의 이름만 출력하세요."
            ),
            input=live_row["input"]
        )
        live_text = response.output_text
        print("응답 ID:", response.id)
        print("응답 상태:", response.status)
        print("생성 텍스트:", live_text)
        print("실제 반환 사용량:", response.usage)
        print("텍스트 존재:", bool(live_text.strip()))
        print("허용 범주:", live_text in ["단일 요건 민원", "다수 요건 민원"])
        live_result = pd.DataFrame([{
            "source_id": live_row["source_id"], "record_id": live_row["record_id"],
            "원문": live_row["input"], "기존_output": live_row["output"],
            "response_id": response.id, "status": response.status,
            "새_API출력": live_text, "검토상태": "원문과 조건 대조 필요"
        }])
        print("live_result는 검토 전 기록입니다. 정확성을 확인한 후 활용하세요.")
    except APIStatusError as error:
        print("요청 실패. HTTP 상태 코드:", error.status_code)
        print("결과를 임의로 채우지 않았습니다. 원인을 확인하고 재요청 여부를 결정하세요.")
    except APIConnectionError:
        print("연결 또는 응답 시간 문제입니다. 연결 상태와 이전 요청 처리 여부를 확인하세요.")
    finally:
        client.close()


### 선택 실행 제공 코드 읽기 — 기본 모드와 실제 호출의 경계

이 셀은 앞 문제의 답을 작성하는 데 필수로 실행할 코드가 아닙니다. `RUN_LIVE_API = False`이면 안내 문장만 출력하고 `else` 안으로 들어가지 않습니다. 이 경우 키 입력·SDK 가져오기·실제 요청을 수행하지 않습니다. 아래 설명은 **기존 제공 코드가 어떤 순서로 동작하는지** 이해하기 위한 것입니다.

| 코드 부분 | 역할 |
|---|---|
| `RUN_LIVE_API` | 실제 호출 부분을 실행할지 정하는 스위치 |
| `LIVE_MODEL_ID` | 실제 이용 가능한 모델 이름을 담는 칸 |
| `if not LIVE_MODEL_ID.strip()` | 모델 이름이 비어 있으면 중단 |
| `os.environ.get("OPENAI_API_KEY")` | 실행 환경에 설정된 키 값을 읽음 |
| `getpass(...)` | 키가 없을 때 일반 출력에 노출하지 않고 입력받음 |
| `live_result = None` | 아직 결과 표가 없다는 초기 상태 |
| `OpenAI(max_retries=0)` | SDK 클라이언트를 만들고 자동 재시도를 끔 |
| `client.responses.create(...)` | 모델·지시·원문을 전달하는 실제 요청 위치 |
| `response.output_text` | SDK 응답에서 생성 텍스트를 읽음 |
| `response.id`, `response.status`, `response.usage` | 응답 식별자·상태·실제 반환 사용량을 각각 확인 |
| `live_result = pd.DataFrame(...)` | 원문·기존 출력·새 응답을 구분한 검토용 표 생성 |

`getpass()`로 입력한 값을 환경변수에 넣는 코드는 현재 실행 환경에서 키를 이용할 수 있게 하는 부분입니다. 키를 노트북 코드나 출력에 적어 공유하는 방식으로 바꾸지 않습니다. 모델 ID 칸에도 키를 입력하지 않습니다.

`try`는 요청과 응답 처리를 시도하는 부분입니다. `except APIStatusError`는 HTTP 오류 응답의 상태 코드를 확인하고, `except APIConnectionError`는 연결·응답 시간 문제를 안내합니다. `finally`는 해당 try 처리 후 클라이언트를 닫는 부분입니다. 이것은 실패를 성공으로 바꾸는 코드가 아닙니다.

현재 제공 코드에서 모델 이름·키 검사와 클라이언트 생성은 `try` 앞에 있습니다. 따라서 모델 이름이 비어 발생한 `ValueError` 같은 준비 단계 오류까지 뒤의 두 `except`가 모두 처리하는 구조는 아닙니다. 출력된 오류가 어느 단계에서 발생했는지 먼저 읽어야 합니다.

자동 재시도를 꺼도 사람이 셀을 다시 실행하면 새로운 요청을 보낼 수 있습니다. 연결 문제로 응답을 받지 못했다면 이전 요청의 처리 여부도 확인합니다. 실패한 경우 `live_result`를 기존 정답으로 채우지 않습니다.

실제 응답을 받아도 `검토상태`는 ‘원문과 조건 대조 필요’로 기록됩니다. 이 표를 만들었다는 것만으로 답의 정확성이 검증된 것은 아닙니다. 상태·텍스트 존재·허용 범주·원문 근거를 모두 확인합니다. 기본 예시의 분류명·응답 ID·토큰 수와 실제 반환값이 반드시 같아야 하는 것도 아닙니다.

검증 범위: 이 보강본은 실제 호출을 끈 상태에서 실행을 확인했습니다. 선택 실행의 실제 응답·사용량·비용은 측정하지 않았습니다. 기존 SDK 구문의 의미는 [OpenAI Python SDK 공식 안내](https://github.com/openai/openai-python)의 요청·응답, 오류 처리, 재시도 설정, 연결 종료 설명과 대조했습니다.
